In [1]:
# import libraries
import pandas as pd
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob

# load the medium sized English model
nlp = spacy.load('en_core_web_md')

nlp.add_pipe('spacytextblob')

In [2]:
# load in the data
df = pd.read_csv('Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv')

**Preprocess the Data:**

In [3]:
# remove missing values from the reviews column
df = df.dropna(subset=['reviews.text'])

In [4]:
# remove stop words

all_reviews = []

for review in df['reviews.text']:

    tidy_review = []
    doc = nlp(str(review).lower().strip())

    for word in doc:
        if not word.is_stop:
            tidy_review.append(word.text)

    all_reviews.append(" ".join(tidy_review))

df['new_reviews'] = all_reviews

**Create a Function for Sentiment Analysis:**

In [20]:
# define a function that takes a product review as input and 
# predicts its sentiment.

def sentiment_analysis(review):
    '''
    Takes a product review as an input.
    Generates a polarity score - if the score is positive,
    return 'positive', if negative return 'negative' and
    else return 'neutral'.
    '''
    doc = nlp(review)

    polarity = doc._.blob.polarity
    
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

**Test the model on sample product reviews:**

In [16]:
# testing the model
for i in range(4):

    sample_review = df['new_reviews'][i]
    
    sentiment_choice = sentiment_analysis(sample_review)
    
    print(f'''
    Review:
    {sample_review}
    Sentiment:
    {sentiment_choice}''')


    Review:
    order 3 item bad quality . missing backup spring pcs aluminum battery work .
    Sentiment:
    Negative

    Review:
    bulk expensive way products like
    Sentiment:
    Negative

    Review:
    duracell price happy .
    Sentiment:
    Positive

    Review:
    work brand batteries better price
    Sentiment:
    Positive


In [19]:
# comparing similarity of product reviews
review_1 = df['new_reviews'][0]
review_2 = df['new_reviews'][1]
review_3 = df['new_reviews'][2]
review_4 = df['new_reviews'][3]

review_1_nlp = nlp(review_1)

similarity_12 = nlp(review_2).similarity(review_1_nlp)
print(f"Review 1 (Negative) and Review 2 (Negative): {similarity_12}")

similarity_13 = nlp(review_3).similarity(review_1_nlp)
print(f"Review 1 (Negative) and Review 3 (Positive): {similarity_13}")

review_3_nlp = nlp(review_3)
similarity_34 = nlp(review_4).similarity(review_3_nlp)
print(f"Review 3 (Positive) and Review 4 (Positive): {similarity_34}")


Review 1 (Negative) and Review 2 (Negative): 0.7760040163993835
Review 1 (Negative) and Review 3 (Positive): 0.7782378792762756
Review 3 (Positive) and Review 4 (Positive): 0.7349037528038025


**Summary:**

The dataset used in this task contains over 28,000 consumer reviews for Amazon products updated between February and April 2019. Examples of columns include basic product information, rating, and manufacturer, but the 'review text' column is what this task focuses on.

Initially in the preprocessing stage, missing values were removed from the reviews column. Each review was then appended to remove stop words, unnecessary whitespace, and turn all text into lowercase.

Testing the function for sentiment analysis was successful, with correct identifications for positive and negative reviews. However, comparing the similarity of product reviews found that two negative reviews and two positive reviews both scored lower than a negative and positive review did together.

This shows that a limitation of spacy's similarity() function is that it relies too heavily on topic rather than tone, for instance just because two reviews mention a battery, it does not mean that they were both happy with the battery and thus it does not mean that the reviews are similar in nature. In contrast, this is a pro of textblob's polarity attribute.

A potential con of the polarity attribute which my sentiment_analysis() function relies upon is lack of nuance. Reviews may depend on context, include sarcasm or vague wording, or include a mixture of positive and negative opinions.